In [18]:
import pandas as pd
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
import os
import math
import time
from tqdm import tqdm

In [5]:
df = pd.read_csv('data/initial_data.csv')
df

,category,title,description,og:url,start_time,end_time,latitude,longitude,image,event_id
0,Charity & Social Causes,"Kissed a Ghoul, Liked It: Diversity in Paranor...",Hear from a panel of authors about writing tra...,https://www.eventbrite.com/e/kissed-a-ghoul-li...,2025-10-26T15:00:00-04:00,2025-10-26 17:00:00-04:00,42.389592,-71.106178,https://cdn.evbuc.com/images/1117003563/286985...,1607191967439
1,Community & Culture,Ultimate New York Food & Culture Tour™ in the ...,Ultimate New York Food & Culture Tour™ in the ...,https://www.eventbrite.com/e/ultimate-new-york...,2014-12-08T14:45:00-05:00,2026-03-15 14:30:00-04:00,40.731660,-74.003140,https://img.evbuc.com/https%3A%2F%2Fcdn.evbuc....,14814904779
2,Community & Culture,NYC Language & Culture Social – Every Friday @...,meet new people from all over the world (and t...,https://www.eventbrite.com/e/nyc-language-cult...,2025-07-25T19:00:00-04:00,2028-09-22 21:00:00-04:00,40.709864,-74.008749,https://cdn.evbuc.com/images/1082246403/283999...,1507737697039
3,Online,NYC Online Speed Dating (56+ age group),Meet someone special and have fun connecting w...,https://www.eventbrite.com/e/nyc-online-speed-...,2025-05-02T20:00:00-04:00,2025-12-26 22:00:00-05:00,40.709778,-74.009827,https://cdn.evbuc.com/images/1016449273/273557...,1339454016019
4,Online,NYC Singles Online Trivia Night - A Fun Speed ...,* New York City Online Singles Trivia Night * ...,https://www.eventbrite.com/e/nyc-singles-onlin...,2025-04-01T21:30:00-04:00,2025-10-14 21:00:00-04:00,40.712775,-74.005973,https://cdn.evbuc.com/images/975367383/1924315...,1269852475939
...,...,...,...,...,...,...,...,...,...,...
313,Comedy & Performance,Bushwick Walk-In Studio Portraits,Capture your best angles & personality for our...,https://www.eventbrite.com/e/bushwick-walk-in-...,2025-09-27T10:00:00-04:00,2025-09-28 21:00:00-04:00,40.706636,-73.928359,https://cdn.evbuc.com/images/1133247373/288987...,1735584683509
314,Comedy & Performance,Great Films You May Never See Film Festival,Come check out some awesome films that may hav...,https://www.eventbrite.com/e/great-films-you-m...,2025-09-28T10:00:00-04:00,2025-09-28 21:00:00-04:00,40.813109,-73.954976,https://cdn.evbuc.com/images/1108238233/332916...,1640765356349
315,Comedy & Performance,SHORT FILM FESTIVAL - LIVE EVENT - NYC INDIE S...,Get ready for a night of indie short films at ...,https://www.eventbrite.com/e/short-film-festiv...,2025-10-03T13:30:00-04:00,2025-10-05 22:00:00-04:00,40.771433,-73.993651,https://cdn.evbuc.com/images/1122737033/576770...,1692733715109
316,Comedy & Performance,Movie Night - Magma,La Soufrière volcano in Guadeloupe threatens e...,https://www.eventbrite.com/e/movie-night-magma...,2025-10-08T18:30:00-04:00,2025-10-08 20:30:00-04:00,40.768862,-73.951532,https://cdn.evbuc.com/images/1108061783/321111...,1640984311249


In [11]:
load_dotenv("secrets.env")
my_key = os.getenv("GEMINI_KEY")

os.environ["GOOGLE_API_KEY"] = my_key
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-001", temperature=1.0)

In [20]:
def generate_broad_tags(description, title):
    tag_prompt = PromptTemplate.from_template("""
       You are assigning tags to events based on their {description} and {title}.
                                            
                                              
        The available tags are 'Comedy & Performance', 'Community & Culture', 'Health & Fitness', 'Food & Drink', 'Nightlife & Parties', 'STEM'                       20
        'Home & Lifestyle', 'Charity & Social Causes', 'Business', 'Music', 'Education', 'Dating' 
                                              
        Return the tags and only the tags with NOTHING else, also assign a maximum of three tags
    """)
    
    tag_chain = tag_prompt | llm
    
    result = tag_chain.invoke({
            "description" : description,
            "title" : title
        })
    
    result = result.content
    return result

In [22]:
batch_size = 20
tags = []

num_batches = math.ceil(len(df) / batch_size)

for i in tqdm(range(num_batches), desc="Processing batches"):
    batch = df.iloc[i*batch_size : (i+1)*batch_size]

    batch_tags = batch.apply(
        lambda row: generate_broad_tags(row['description'], row['title']),
        axis=1
    ).tolist()
    
    tags.extend(batch_tags)
    
    if i < num_batches - 1:
        time.sleep(30)

df['category_ai'] = tags

Processing batches:  12%|█▎        | 2/16 [08:46<1:03:35, 272.51s/it]Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15
Please retry in 32.583522402s. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 32


ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 200
Please retry in 23.70650285s. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 200
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 23
}
]